In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

print("--- Phase 5: Production Deployment Simulator ---")

# 1. Load the AI Brains and Production Metadata
print("Loading saved AI models and pipeline metadata...")
try:
    scaler = joblib.load('../models/clv_scaler.joblib')
    kmeans_model = joblib.load('../models/clv_kmeans_model.joblib')
    persona_map = joblib.load('../models/persona_map.joblib') 
    xgb_model = joblib.load('../models/future_spend_xgboost_model.joblib') 
    expected_features = joblib.load('../models/predictive_feature_columns.joblib') 
    print("✅ All systems online. Ready for new customers.\n")
except FileNotFoundError as e:
    print(f"❌ ERROR: Missing model file. Did you run the previous notebooks? Details: {e}")
    raise

# 2. A brand new, unseen customer profile for deployment simulation
new_customer_data = {
    'active_duration_days': [360],
    'avg_days_between_swipes': [0.3],    
    'frequency': [1200],
    'monetary_sum': [75000.00],
    'monetary_mean': [62.50],
    'monetary_max': [2500.00],            
    'monetary_std': [150.00]              
}

new_customer_df = pd.DataFrame(new_customer_data)

# Production Realism: Input Validation
if (new_customer_df < 0).any().any():
    raise ValueError("❌ Data Error: Customer features cannot be negative.")

print("--- New Customer Profile ---")
display(new_customer_df)

# 3. Step A: Scale the new customer's data and find their Persona
scaled_data = scaler.transform(new_customer_df)
scaled_df = pd.DataFrame(scaled_data, columns=new_customer_df.columns)

predicted_cluster_id = kmeans_model.predict(scaled_df)[0]

# Translate the ID into the human-readable string using our saved dynamic map
persona_name = persona_map[predicted_cluster_id]
print(f"\n🧠 K-Means AI says: This customer belongs to the [{persona_name}] Persona.")

# 4. Step B: BULLETPROOF FEATURE ALIGNMENT
# Create an empty dataframe with the exact columns XGBoost expects and fill it with 0s
final_data = pd.DataFrame(0.0, index=[0], columns=expected_features)

# Map our customer's numerical data into the right slots
for col in new_customer_df.columns:
    if col in expected_features:
        final_data.at[0, col] = new_customer_df.at[0, col]

# Dynamically trigger the correct One-Hot Encoded Persona
persona_col = f'Seg_{persona_name}'
if persona_col in expected_features:
    final_data.at[0, persona_col] = 1.0

# 5. Step C: Predict the Spend Growth Rate and Dollar Value
predicted_log_ratio = xgb_model.predict(final_data)[0]

# Convert the Log Ratio back to a Real Percentage
predicted_real_ratio = np.exp(predicted_log_ratio)
growth_percentage = (predicted_real_ratio - 1.0) * 100

# Convert the ratio into actual forecasted DOLLARS
historical_spend = new_customer_df.at[0, 'monetary_sum']
predicted_future_spend = predicted_real_ratio * historical_spend

print(f"\n🔮 XGBoost AI Forecast:")
print(f"   ▶ Growth Ratio: {predicted_real_ratio:.2f} ({growth_percentage:+.1f}%)")
print(f"   ▶ Predicted 2020 Spend: ${predicted_future_spend:,.2f}")

if predicted_real_ratio < 0.95:
    print("\n🚨 DECISION: This customer is predicted to drop their spending. Trigger retention marketing!")
elif predicted_real_ratio > 1.05:
    print("\n📈 DECISION: This customer is predicted to grow their spend. Offer premium upgrades.")
else:
    print("\n⚖️ DECISION: This customer's spending is predicted to remain stable. Monitor engagement.")

--- Phase 5: Production Deployment Simulator ---
Loading saved AI models and pipeline metadata...
✅ All systems online. Ready for new customers.

--- New Customer Profile ---


,active_duration_days,avg_days_between_swipes,frequency,monetary_sum,monetary_mean,monetary_max,monetary_std
0,360,0.3,1200,75000.0,62.5,2500.0,150.0



🧠 K-Means AI says: This customer belongs to the [Engaged Regulars] Persona.

🔮 XGBoost AI Forecast:
   ▶ Growth Ratio: 0.36 (-63.6%)
   ▶ Predicted 2020 Spend: $27,335.45

🚨 DECISION: This customer is predicted to drop their spending. Trigger retention marketing!


In [2]:
pip install streamlit

  Using cached streamlit-1.57.0-py3-none-any.whl (9.2 MB)
  Using cached altair-6.1.0-py3-none-any.whl (796 kB)
  Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
  Using cached cachetools-7.1.1-py3-none-any.whl (16 kB)
  Using cached click-8.3.3-py3-none-any.whl (110 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl (212 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl (11.3 MB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl (437 kB)
  Using cached pyarrow-24.0.0-cp311-cp311-win_amd64.whl (27.3 MB)
  Using cached starlette-1.0.0-py3-none-any.whl (72 kB)
  Using cached uvicorn-0.46.0-py3-none-any.whl (70 kB)
  Using cached httptools-0.7.1-cp311-cp311-win_amd64.whl (86 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
  Using cached narwhals-2.21.0-py3-none-any.whl (451 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
